In [2]:
import pandas as pd

# CSV ファイルの読み込み（例: Titanic データ）
df = pd.read_csv("train.csv")

# データの確認
print(df.head())


   id     Podcast_Name Episode_Title  Episode_Length_minutes       Genre  \
0   0  Mystery Matters    Episode 98                     NaN  True Crime   
1   1    Joke Junction    Episode 26                  119.80      Comedy   
2   2   Study Sessions    Episode 16                   73.90   Education   
3   3   Digital Digest    Episode 45                   67.17  Technology   
4   4      Mind & Body    Episode 86                  110.51      Health   

   Host_Popularity_percentage Publication_Day Publication_Time  \
0                       74.81        Thursday            Night   
1                       66.95        Saturday        Afternoon   
2                       69.97         Tuesday          Evening   
3                       57.22          Monday          Morning   
4                       80.07          Monday        Afternoon   

   Guest_Popularity_percentage  Number_of_Ads Episode_Sentiment  \
0                          NaN            0.0          Positive   
1           

In [3]:
df = df.drop(columns=["Name","Ticket","Cabin"], axis=1)

df["Sex"] = df["Sex"].map({"male": 0, "female": 1})
df["Embarked"] = df["Embarked"].map({"C": 0, "Q": 1, "S": 2})

In [4]:
print(df.head())

   PassengerId  Survived  Pclass  Sex   Age  SibSp  Parch     Fare  Embarked
0            1         0       3    0  22.0      1      0   7.2500       2.0
1            2         1       1    1  38.0      1      0  71.2833       0.0
2            3         1       3    1  26.0      0      0   7.9250       2.0
3            4         1       1    1  35.0      1      0  53.1000       2.0
4            5         0       3    0  35.0      0      0   8.0500       2.0


In [5]:
df = df.fillna(df.median())

In [6]:
X = df.drop("Survived", axis=1)
y = df["Survived"]

In [11]:
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)


In [22]:
import lightgbm as lgb
from lightgbm import LGBMClassifier
from lightgbm.callback import early_stopping

# ハイパーパラメータの設定
params = {
    "objective": "binary",  # 二値分類
    "metric": "binary_error",
    "boosting_type": "gbdt",
    "num_leaves": 31,
    "learning_rate": 0.05,
    "feature_fraction": 0.9,
    "n_estimators": 100,
    "max_depth": -1,
    "colsample_bytree": 0.9,
    "verbose": 1  # 進行状況を表示
}

model = LGBMClassifier(**params)

# 学習（early_stopping を適用）
model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    callbacks=[early_stopping(10)],  # early_stopping を callbacks に設定
)


[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=0.9 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=0.9 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Info] Number of positive: 268, number of negative: 444
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000172 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 432
[LightGBM] [Info] Number of data points in the train set: 712, number of used features: 8
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=0.9 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.376404 -> initscore=-0.504838
[LightGBM] [Info] Start training from score -0.504838
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 10 rounds
[Light

LGBMClassifier(colsample_bytree=0.9, feature_fraction=0.9, learning_rate=0.05,
               metric='binary_error', objective='binary', verbose=1)

In [23]:
from sklearn.metrics import accuracy_score

# 予測
y_pred = model.predict(X_valid)
y_pred = (y_pred > 0.5).astype(int)  # 0.5 以上を 1、それ以外を 0 に変換

# 精度の計算
accuracy = accuracy_score(y_valid, y_pred)
print(f"Accuracy: {accuracy:.4f}")


[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=0.9 will be ignored. Current value: feature_fraction=0.9
Accuracy: 0.7989


In [30]:
# test.csv を読み込む
test_df = pd.read_csv('org/test.csv')

# 必要に応じて前処理（例えば特徴量の選択やスケーリング）を行う
# X_test = test_data[必要なカラム]  # 予測に必要な特徴量のみを選択する
print(test_df.head())

   PassengerId  Pclass                                          Name     Sex  \
0          892       3                              Kelly, Mr. James    male   
1          893       3              Wilkes, Mrs. James (Ellen Needs)  female   
2          894       2                     Myles, Mr. Thomas Francis    male   
3          895       3                              Wirz, Mr. Albert    male   
4          896       3  Hirvonen, Mrs. Alexander (Helga E Lindqvist)  female   

    Age  SibSp  Parch   Ticket     Fare Cabin Embarked  
0  34.5      0      0   330911   7.8292   NaN        Q  
1  47.0      1      0   363272   7.0000   NaN        S  
2  62.0      0      0   240276   9.6875   NaN        Q  
3  27.0      0      0   315154   8.6625   NaN        S  
4  22.0      1      1  3101298  12.2875   NaN        S  


In [31]:
test_df = test_df.drop(columns=["Name","Ticket","Cabin"], axis=1)
test_df["Sex"] = test_df["Sex"].map({"male": 0, "female": 1})
test_df["Embarked"] = test_df["Embarked"].map({"C": 0, "Q": 1, "S": 2})

In [32]:
print(test_df.head())

   PassengerId  Pclass  Sex   Age  SibSp  Parch     Fare  Embarked
0          892       3    0  34.5      0      0   7.8292         1
1          893       3    1  47.0      1      0   7.0000         2
2          894       2    0  62.0      0      0   9.6875         1
3          895       3    0  27.0      0      0   8.6625         2
4          896       3    1  22.0      1      1  12.2875         2


In [33]:
predictions = model.predict(test_df)  # test_data は前処理後のデータ

[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=0.9 will be ignored. Current value: feature_fraction=0.9


In [37]:
import pandas as pd
# ID（例えば 'PassengerId'）のカラム名が 'ID' であると仮定
submission = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],  # もとのIDを取得
    'Survived': predictions  # 予測結果
})

# CSV に保存
submission.to_csv("submission.csv", index=False)
